In [ ]:
import datetime

import atproto
from sqlalchemy import select
from sqlalchemy.orm import selectinload

import config
from core.database import session_scope
from core.database.models import ThemeVoteRecord
from core.interactions.votes import current_theme_vote, get_theme_vote

In [ ]:
with session_scope() as session:
    vote = current_theme_vote(session)

if vote is None:
    raise Exception('no current theme vote right now - nothing to check')

print(f'current vote {vote.id} ({vote.type.value})')
print(f'  voting: {vote.vote_start_date} -> {vote.vote_end_date}')
for option in vote.options:
    print(f'    - {option.theme_name}: likes so far = {option.likes}')

In [6]:
client = atproto.Client()
client.login(config.ATPROTO_CLIENT_USERNAME, config.ATPROTO_CLIENT_PASSWORD)

ProfileViewDetailed(did='did:plc:lk63sepwkbzg6cm67ws7ihq4', handle='randomcolormemes.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:lk63sepwkbzg6cm67ws7ihq4/bafkreiedgvyny4xjecczfjvk3o5etogdhff27fj6ncoewxutyp3weyszi4', banner='https://cdn.bsky.app/img/banner/plain/did:plc:lk63sepwkbzg6cm67ws7ihq4/bafkreifzzgm5qaus6x7ozsdzvtki2mnlnu7bpxhm3bqpxhhwiraf5dn6sa', created_at='2026-01-24T09:35:14.545Z', debug=None, description='Posts a random color every hour.', display_name='Random Color Memes', followers_count=22, follows_count=0, indexed_at='2026-06-08T05:47:07.865Z', joined_via_starter_pack=None, labels=[], pinned_post=Main(cid='bafyreihjnekfvxciwwr6jpjzhvmzogk3e43oxh7

In [7]:
uris = [o.comment_uri for o in vote.options if o.comment_uri]
if not uris:
    raise SystemExit('this vote has no posted option comments to check')

posts = client.get_posts(uris).posts
likes_by_uri = {post.uri: (post.like_count or 0) for post in posts}

for option in vote.options:
    count = likes_by_uri.get(option.comment_uri)
    print(f'  {option.theme_name}: {count if count is not None else "not found"}')

  Onyx: not found
  Pikachu: not found
  Monochrome: not found
  Winter: not found
  Synthwave: not found


In [8]:
now = datetime.datetime.now(datetime.timezone.utc)

with session_scope() as session:
    record = session.scalar(
        select(ThemeVoteRecord)
        .where(ThemeVoteRecord._id == vote.id)
        .options(selectinload(ThemeVoteRecord.options))
    )

    for option in record.options:
        fetched = likes_by_uri.get(option.comment_uri)
        if fetched is not None:
            option.likes = fetched
        option._checked_date = now
        print(f'  {option.theme_name}: {option.likes} likes')

    record._checked_date = now

print('saved')

  Onyx: None likes
  Pikachu: None likes
  Monochrome: None likes
  Winter: None likes
  Synthwave: None likes
saved


In [9]:
with session_scope() as session:
    record = session.scalar(
        select(ThemeVoteRecord)
        .where(ThemeVoteRecord._id == vote.id)
        .options(selectinload(ThemeVoteRecord.options))
    )
    print(f'vote {record.id}  checked={record._checked_date}')
    for option in record.options:
        print(f'  {option.theme_name:<16} likes={option.likes}  checked={option._checked_date}')

with session_scope() as session:
    final = get_theme_vote(session, vote.id)
print('current leader:', final.winner.theme_name if final.winner else None)

vote 1  checked=2026-06-14 07:25:48.698588
  Onyx             likes=None  checked=2026-06-14 07:25:48.698588
  Pikachu          likes=None  checked=2026-06-14 07:25:48.698588
  Monochrome       likes=None  checked=2026-06-14 07:25:48.698588
  Winter           likes=None  checked=2026-06-14 07:25:48.698588
  Synthwave        likes=None  checked=2026-06-14 07:25:48.698588
current leader: None
